# Kulturologija i komunikacijski/društveni problemi: praktični primjer

**Povezano s sadržajem:** [1. Uvod — Kulturologija i komunikacijski/društveni problemi](../content/01_uvod_drustvene_mreze_teorija_grafova.md#kulturologija-i-komunikacijskidruštveni-problemi)

Mrežni pristup nije samo tehnička metoda; pogodan je za pitanja **identiteta, moći i širenja informacija** u kulturnim i komunikacijskim kontekstima. U ovom primjeru analiziramo **mrežu likova** iz romana *Les Misérables* (Victor Hugo): tko je „središnji”, tko povezuje grupe („most”), kako struktura odražava društvenu moć i pristup informacijama u narativu.

**Što radimo:**
- Učitavamo podatke o supojavljivanju likova (veze = zajedničke scene) u `examples/data`.
- Računamo mjere centralnosti (stupanj, betweenness) i vizualiziramo ih pomoću **pandas** i **plotly**.
- Interpretiramo rezultate u smislu kulturologije: položaj u mreži → moć, identitet, mostovi između grupa.

## Korak 1: Priprema okruženja i putanja do podataka

Koristimo **pandas** za tablice, **networkx** za graf i mjere, **plotly** za interaktivnu vizualizaciju. Podaci se spremaju u `examples/data`.

In [ ]:
import pandas as pd
import networkx as nx
from pathlib import Path
try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    import matplotlib.pyplot as plt

# Putanja do examples/data (radna mapa = korijen projekta ili examples)
cwd = Path.cwd()
if (cwd / "examples" / "data").exists():
    DATA_DIR = cwd / "examples" / "data"
elif (cwd / "data").exists():
    DATA_DIR = cwd / "data"
else:
    DATA_DIR = cwd / "examples" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Putanja do podataka:", DATA_DIR)

## Korak 2: Preuzimanje ili učitavanje podataka

Ako u `examples/data` već postoje datoteke `les_miserables_edges.csv` i `les_miserables_nodes.csv`, učitavamo ih. Inače gradimo graf pomoću NetworkX-a (*Les Misérables* mreža) i spremamo ga u CSV kako bi sljedeći put podaci bili na disku.

In [ ]:
edges_path = DATA_DIR / "les_miserables_edges.csv"
nodes_path = DATA_DIR / "les_miserables_nodes.csv"

if edges_path.exists() and nodes_path.exists():
    edges_df = pd.read_csv(edges_path)
    nodes_df = pd.read_csv(nodes_path)
    print("Učitano s diska:", len(nodes_df), "čvorova,", len(edges_df), "veza.")
else:
    # Ugrađena mreža likova iz Les Misérables (NetworkX)
    G_src = nx.les_miserables_graph()
    edges_df = pd.DataFrame([(u, v, d.get("weight", 1)) for u, v, d in G_src.edges(data=True)],
                             columns=["source", "target", "weight"])
    nodes_df = pd.DataFrame([{"id": n, "name": n} for n in G_src.nodes()])
    edges_df.to_csv(edges_path, index=False)
    nodes_df.to_csv(nodes_path, index=False)
    print("Spremljeno u", DATA_DIR)

display(edges_df.head(10))
display(nodes_df.head(10))

## Korak 3: Izgradnja grafa i osnovne statistike

Gradimo **neusmjereni težinski graf**: čvorovi = likovi, veze = supojavljivanje u sceni, težina = broj zajedničkih scena. *(Definicije: [pogl. 1 — Čvorovi, veze, težinski graf](../content/01_uvod_drustvene_mreze_teorija_grafova.md#definicije).)*

In [ ]:
G = nx.Graph()
for _, row in edges_df.iterrows():
    G.add_edge(row["source"], row["target"], weight=row["weight"])

n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
density = nx.density(G)
print(f"Čvorova: {n_nodes}, veza: {n_edges}, gustoća: {density:.4f}")

# Tablica stupnjeva (broj veza po liku)
degree = dict(G.degree())
stats_df = pd.DataFrame([{"Lik": n, "Stupanj": d} for n, d in degree.items()]).sort_values("Stupanj", ascending=False)
display(stats_df.head(15))

## Korak 4: Mjere centralnosti — tko je „središnji”, tko „most”?

Za kulturološku interpretaciju računamo:
- **Stupanj (degree):** koliko je lik izravno povezan s drugima → „vidljivost” u narativu.
- **Betweenness:** koliko puta lik leži na najkraćem putu između drugih parova → **most** između grupa, posrednik u toku informacija.

*(Mjere centralnosti: [pogl. 1 — Osnove teorije grafova](../content/01_uvod_drustvene_mreze_teorija_grafova.md#osnove-teorije-grafova-i-primjena-u-društvenim-mrežama).)*

In [ ]:
betweenness = nx.betweenness_centrality(G, weight="weight")
degree_cent = nx.degree_centrality(G)

centrality_df = pd.DataFrame([
    {
        "Lik": n,
        "Stupanj": degree[n],
        "Degree centralnost": round(degree_cent[n], 4),
        "Betweenness": round(betweenness[n], 4),
    }
    for n in G.nodes()
])
centrality_df = centrality_df.sort_values("Betweenness", ascending=False)
display(centrality_df.head(15))

## Korak 5: Interpretacija za kulturologiju i komunikaciju

| Mjera | Kulturološka/komunikacijska interpretacija |
|-------|------------------------------------------|
| **Visok stupanj** | Lik je u mnogo scena s drugima → „središnji” u priči, velika društvena vidljivost. |
| **Visok betweenness** | Lik povezuje različite grupe likova (most) → posrednik u širenju informacija, moć nad tokovima. |

U *Les Misérables* likovi s visokim betweenness-om često su oni koji povezuju društvene sfere (npr. grad–selo, zakon–kriminal). To ilustrira kako **položaj u mreži** odražava društvenu moć i pristup informacijama — ključno za kulturološke i komunikacijske studije.

In [ ]:
# Primjer: top 5 po betweenness („mostovi”)
top_between = centrality_df.nlargest(5, "Betweenness")
print("Likovi s najvećom ulogom 'mosta' (betweenness):")
print(top_between[["Lik", "Betweenness", "Stupanj"]].to_string(index=False))

## Korak 6: Interaktivna vizualizacija (Plotly)

Crtamo mrežu u **plotly**: čvorovi su likovi, veličina čvora proporcionalna **betweenness** (naglašavamo „mostove”), boja ili veličina može odražavati stupanj. Layout: `nx.spring_layout` za pozicije.

In [ ]:
pos = nx.spring_layout(G, seed=42, k=0.5)
node_names = list(G.nodes())
node_between = [betweenness[n] for n in node_names]
node_degree = [degree[n] for n in node_names]
node_size = (8 + 80 * pd.Series(node_between)).tolist()

if HAS_PLOTLY:
    node_x = [pos[n][0] for n in node_names]
    node_y = [pos[n][1] for n in node_names]
    edge_x, edge_y = [], []
    for u, v in G.edges():
        x0, y0, x1, y1 = pos[u][0], pos[u][1], pos[v][0], pos[v][1]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines", line=dict(width=0.5, color="#888"), hoverinfo="none"))
    fig.add_trace(go.Scatter(
        x=node_x, y=node_y,
        mode="markers+text",
        text=node_names,
        textposition="top center",
        marker=dict(size=node_size, color=node_between, colorscale="Viridis", showscale=True, colorbar=dict(title="Betweenness")),
        hovertext=[f"{n}<br>Betweenness: {b:.3f}<br>Stupanj: {d}" for n, b, d in zip(node_names, node_between, node_degree)],
        hoverinfo="text",
        textfont=dict(size=8),
    ))
    fig.update_layout(
        title="Les Misérables: mreža likova (veličina i boja ∝ betweenness — mostovi između grupa)",
        showlegend=False,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        template="plotly_white",
        height=600,
    )
    fig.show()
else:
    plt.figure(figsize=(12, 8))
    nx.draw(G, pos, nodelist=node_names, node_size=node_size, node_color=node_between, cmap="viridis",
            with_labels=True, font_size=6, font_weight="bold", width=0.5, edge_color="#888")
    plt.title("Les Misérables: mreža likova (veličina i boja ∝ betweenness)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

## Korak 7: Pregled i povezivanje sa sadržajem

U ovoj bilježnici:
- **Podaci:** `examples/data/les_miserables_edges.csv`, `les_miserables_nodes.csv`.
- **Alati:** pandas (tablice), networkx (graf, centralnost), plotly (interaktivni graf).
- **Kulturološka pitanja:** identitet i moć (stupanj), mostovi i širenje informacija (betweenness), grupe u narativu.

**Sadržaj ↔ kod:** Teorijski okvir u [pogl. 1 — Kulturologija i komunikacijski/društveni problemi](../content/01_uvod_drustvene_mreze_teorija_grafova.md#kulturologija-i-komunikacijskidruštveni-problemi). Tablica „Povezivanje sadržaj ↔ kod” na kraju tog dokumenta.